In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Data Loading
import pandas as pd

# File Paths
train_path = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
test_path = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

# Load the datasets
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display first 5 rows of training data
print(train_df.head())


In [ ]:
# ============================================
# EDA for Kaggle House Prices (train_df) — FIXED
# - Cleans up FutureWarnings
# - Handles ±inf -> NaN
# - Uses observed=True in groupby order calcs
# - Adds clear titles and axis labels
# ============================================

# If you don't already have train_df, uncomment the loader below.
# import pandas as pd
# TRAIN_PATH = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
# train_df = pd.read_csv(TRAIN_PATH)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# -------------------------------
# Silence specific deprecation warnings
# -------------------------------
warnings.filterwarnings("ignore", message=".*use_inf_as_na.*")
warnings.filterwarnings("ignore", message=".*default of observed=False is deprecated.*")

# -------------------------------
# Basic cleaning: convert ±inf to NaN in numeric columns
# -------------------------------
if 'train_df' not in globals():
    raise RuntimeError("Please define `train_df` or uncomment the CSV loader at the top.")

num_cols_all = train_df.select_dtypes(include=[np.number]).columns
train_df[num_cols_all] = train_df[num_cols_all].replace([np.inf, -np.inf], np.nan)

# -------------------------------
# 0) Basic structure & statistics
# -------------------------------
print("=== train_df.info() ===")
train_df.info()
print("\n=== train_df.describe(include='all') ===")
display(train_df.describe(include='all').transpose())

# -------------------------------
# 1) Histograms of SalePrice
#    (raw and log-transformed)
# -------------------------------
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.histplot(train_df["SalePrice"], bins=40, kde=True)
plt.title("Distribution of SalePrice (Raw)")
plt.xlabel("SalePrice ($)")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
saleprice_pos = train_df["SalePrice"].clip(lower=1)  # ensure > 0 for log
sns.histplot(np.log1p(saleprice_pos), bins=40, kde=True)
plt.title("Distribution of SalePrice (log1p transformed)")
plt.xlabel("log1p(SalePrice)")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 2) Heatmap: correlations among the top 10 numeric
#    features most related to SalePrice
# -------------------------------------------------------
numeric_df = train_df.select_dtypes(include=[np.number]).copy()

# In case there are columns with all-NaN after inf replacement, drop them
numeric_df = numeric_df.dropna(axis=1, how="all")

# Compute correlations and choose top 10 (by |corr|) with SalePrice
corr_series = numeric_df.corr(numeric_only=True)["SalePrice"].drop(labels=["SalePrice"])
top10_features = (
    corr_series.reindex(corr_series.abs().sort_values(ascending=False).index)
    .head(10)
    .index.tolist()
)

cols_for_heatmap = ["SalePrice"] + top10_features
corr_matrix = numeric_df[cols_for_heatmap].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="crest",
    square=True,
    cbar_kws={"shrink": 0.8}
)
plt.title("Correlation Heatmap: SalePrice and Top 10 Related Numeric Features")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 3) Boxplots for categorical variables vs SalePrice
#    Examples: Neighborhood and HouseStyle
# -------------------------------------------------------
def plot_cat_boxplot(df, cat_col, y_col="SalePrice", figsize=(14, 6), rotate=70):
    if cat_col not in df.columns:
        print(f"Skipping '{cat_col}': column not found.")
        return
    # Ensure categorical dtype for consistency
    df = df.copy()
    df[cat_col] = df[cat_col].astype("category")

    # Order categories by median SalePrice; observed=True avoids pandas futurewarning
    order = (
        df.groupby(cat_col, observed=True)[y_col]
        .median()
        .sort_values()
        .index
    )

    plt.figure(figsize=figsize)
    sns.boxplot(
        data=df,
        x=cat_col,
        y=y_col,
        order=order,
        showfliers=False
    )
    plt.title(f"{y_col} by {cat_col}")
    plt.xlabel(f"{cat_col} (sorted by median {y_col})")
    plt.ylabel(f"{y_col} ($)")
    plt.xticks(rotation=rotate, ha="right")
    plt.tight_layout()
    plt.show()

# Neighborhood vs SalePrice
plot_cat_boxplot(train_df, "Neighborhood", "SalePrice", figsize=(14, 6), rotate=70)

# HouseStyle vs SalePrice (optional second categorical example)
plot_cat_boxplot(train_df, "HouseStyle", "SalePrice", figsize=(12, 6), rotate=45)

# -------------------------------
# (Optional) Boxplots on log scale
# -------------------------------
# Example: Neighborhood with log y-scale
# plt.figure(figsize=(14,6))
# order = (
#     train_df.groupby("Neighborhood", observed=True)["SalePrice"]
#     .median()
#     .sort_values()
#     .index
# )
# sns.boxplot(
#     data=train_df,
#     x="Neighborhood",
#     y="SalePrice",
#     order=order,
#     showfliers=False
# )
# plt.yscale("log")
# plt.title("SalePrice by Neighborhood (Log Scale)")
# plt.xlabel("Neighborhood (sorted by median SalePrice)")
# plt.ylabel("SalePrice (log scale)")
# plt.xticks(rotation=70, ha="right")
# plt.tight_layout()
# plt.show()


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np
import pandas as pd

TRAIN_PATH = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
TEST_PATH  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

y = train["SalePrice"].copy()
X = train.drop(columns=["SalePrice", "Id"])
X_test = test.drop(columns=["Id"])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

numeric_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  # dense OK for this dataset size
])

categorical_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # Make encoder DENSE to avoid sparse_cg path + silence FutureWarning
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Force DENSE output from the column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pre, num_cols),
        ("cat", categorical_pre, cat_cols),
    ],
    sparse_threshold=0.0   # ensures a dense numpy array output
)

alphas = np.logspace(-3, 4, 20)
ridgecv = RidgeCV(alphas=alphas, cv=5)

model = TransformedTargetRegressor(
    regressor=Pipeline(steps=[
        ("prep", preprocessor),
        ("ridge", ridgecv)
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)
rmse_scorer = make_scorer(rmse, greater_is_better=False)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, scoring=rmse_scorer, cv=cv, n_jobs=-1)
print(f"5-fold CV RMSE: {(-cv_scores).mean():.2f} ± {(-cv_scores).std():.2f}")

model.fit(X, y)
best_alpha = model.regressor_.named_steps["ridge"].alpha_
print(f"Best Ridge alpha: {best_alpha:.5f}")

test_pred = model.predict(X_test)
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": test_pred})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, make_scorer
import numpy as np

# --- Cross-validated metrics (preferred) ---
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# R^2
cv_r2 = cross_val_score(model, X, y, scoring="r2", cv=cv, n_jobs=-1)

# MSE (negative values returned; take negative to get MSE)
cv_mse_neg = cross_val_score(model, X, y,
                             scoring="neg_mean_squared_error",
                             cv=cv, n_jobs=-1)
cv_mse = -cv_mse_neg
cv_rmse = np.sqrt(cv_mse)

print("=== 5-fold CV Metrics (original $ scale) ===")
print(f"R^2:  {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"MSE:  {cv_mse.mean():.2f} ± {cv_mse.std():.2f}")
print(f"RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}")

# --- Fit on full train and report in-sample metrics (optimistic) ---
model.fit(X, y)
y_pred_in = model.predict(X)  # TransformedTargetRegressor returns predictions on original scale

r2_in = r2_score(y, y_pred_in)
mse_in = mean_squared_error(y, y_pred_in)
rmse_in = mean_squared_error(y, y_pred_in, squared=False)

print("\n=== In-sample Metrics (fit on all training data) ===")
print(f"R^2:  {r2_in:.4f}")
print(f"MSE:  {mse_in:.2f}")
print(f"RMSE: {rmse_in:.2f}")


In [ ]:
# =========================================
# Kaggle House Prices — RidgeCV baseline
# NaN-safe features + dense OHE (no kernel crash)
# =========================================
import warnings
warnings.filterwarnings("ignore", message=".*use_inf_as_na.*")
warnings.filterwarnings("ignore", message=".*default of observed=False is deprecated.*")
warnings.filterwarnings("ignore", message="`sparse` was renamed to `sparse_output`", category=FutureWarning)

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

# ---- paths (Kaggle) ----
TRAIN_PATH = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
TEST_PATH  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# ------------------------
# Helpers
# ------------------------
def make_ohe(dtype=np.float32):
    """
    Create OneHotEncoder that ALWAYS returns dense output.
    Dense is small enough for this dataset and avoids sparse->dense
    surprises inside RidgeCV/CV that can cause OOM.
    """
    try:
        # sklearn >= 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=dtype)
    except TypeError:
        # sklearn <= 1.1
        return OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=dtype)

def safe_get(df, col, default=np.nan):
    return df[col] if col in df.columns else pd.Series(default, index=df.index)

def to_num(s, fill=None):
    out = pd.to_numeric(s, errors="coerce")
    if fill is not None:
        out = out.fillna(fill)
    return out

# ------------------------
# Feature Engineering (NaN/Inf-safe)
# ------------------------
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Coerce bases to numeric
    TotalBsmtSF = to_num(safe_get(out, "TotalBsmtSF"), fill=0)
    FirstFlrSF  = to_num(safe_get(out, "1stFlrSF"), fill=0)
    SecondFlrSF = to_num(safe_get(out, "2ndFlrSF"), fill=0)
    FullBath    = to_num(safe_get(out, "FullBath"), fill=0)
    HalfBath    = to_num(safe_get(out, "HalfBath"), fill=0)
    BsmtFull    = to_num(safe_get(out, "BsmtFullBath"), fill=0)
    BsmtHalf    = to_num(safe_get(out, "BsmtHalfBath"), fill=0)
    YrSold      = to_num(safe_get(out, "YrSold"))
    YearBuilt   = to_num(safe_get(out, "YearBuilt"))
    YearRemod   = to_num(safe_get(out, "YearRemodAdd"))
    OpenPorchSF = to_num(safe_get(out, "OpenPorchSF"), fill=0)
    EnclosedPor = to_num(safe_get(out, "EnclosedPorch"), fill=0)
    ThreeSsn    = to_num(safe_get(out, "3SsnPorch"), fill=0)
    ScreenPorch = to_num(safe_get(out, "ScreenPorch"), fill=0)
    WoodDeckSF  = to_num(safe_get(out, "WoodDeckSF"), fill=0)
    PoolArea    = to_num(safe_get(out, "PoolArea"))
    GarageArea  = to_num(safe_get(out, "GarageArea"))
    GarageYrBlt = to_num(safe_get(out, "GarageYrBlt"))
    GrLivArea   = to_num(safe_get(out, "GrLivArea"))
    LotArea     = to_num(safe_get(out, "LotArea"))
    TotRms      = to_num(safe_get(out, "TotRmsAbvGrd"), fill=0)
    OverallQual = to_num(safe_get(out, "OverallQual"), fill=0)
    OverallCond = to_num(safe_get(out, "OverallCond"), fill=0)

    # Totals & counts
    out["TotalSF"] = TotalBsmtSF + FirstFlrSF + SecondFlrSF
    out["TotalBaths"] = FullBath + 0.5 * HalfBath + BsmtFull + 0.5 * BsmtHalf

    # Ages & flags
    out["HouseAge"] = (YrSold - YearBuilt).fillna(0)
    out["RemodAge"] = (YrSold - YearRemod).fillna(0)
    out["IsRemodeled"] = ((YearRemod - YearBuilt).fillna(-1).gt(0)).astype(int)
    out["IsNew"] = (YrSold == YearBuilt).astype(int)

    # Porches/decks aggregate
    out["TotalPorchSF"] = OpenPorchSF + EnclosedPor + ThreeSsn + ScreenPorch + WoodDeckSF

    # Binary indicators
    out["HasPool"]     = PoolArea.fillna(0).gt(0).astype(int)
    out["Has2ndFloor"] = SecondFlrSF.fillna(0).gt(0).astype(int)
    out["HasBsmt"]     = TotalBsmtSF.fillna(0).gt(0).astype(int)
    out["HasGarage"]   = GarageArea.fillna(0).gt(0).astype(int)

    # Interactions / ratios
    out["QualCond"] = OverallQual * OverallCond
    out["Qual_LnSF"] = OverallQual * np.log1p(GrLivArea.fillna(0))
    out["BathsPerRoom"] = np.divide(out["TotalBaths"], (TotRms + 1.0))

    # LotRatio = GrLivArea / LotArea (safe divide)
    denom = LotArea.values
    numer = GrLivArea.fillna(0).values
    valid = np.isfinite(denom) & (denom > 0)
    lotratio = np.full_like(numer, np.nan, dtype=float)
    lotratio[valid] = numer[valid] / denom[valid]
    out["LotRatio"] = lotratio

    # GarageAge safe construction
    garage_age = (YrSold - GarageYrBlt).values.astype(float)
    mask_g_nan = ~np.isfinite(GarageYrBlt.values)
    mask_has_g = GarageArea.fillna(0).values > 0
    # fill missing garage build years: use house age if garage exists; else 0
    fallback = (YrSold - YearBuilt).fillna(0).values
    garage_age[np.where(mask_g_nan & mask_has_g)] = fallback[np.where(mask_g_nan & mask_has_g)]
    garage_age[np.where(mask_g_nan & ~mask_has_g)] = 0.0
    garage_age[~np.isfinite(garage_age)] = 0.0
    out["GarageAge"] = garage_age

    # Replace inf with NaN (imputer will handle)
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)

    return out

def build_neighborhood_target_stat(train_df: pd.DataFrame, m: float = 50.0):
    """Smoothed target stat for Neighborhood (no leakage)."""
    if "Neighborhood" not in train_df.columns:
        return {}
    overall = train_df["SalePrice"].mean()
    gb = train_df.groupby("Neighborhood")["SalePrice"].agg(["mean", "count"])
    smoothed = (gb["count"] * gb["mean"] + m * overall) / (gb["count"] + m)
    return smoothed.to_dict()

# ---- apply features ----
train_fe = add_engineered_features(train)
test_fe  = add_engineered_features(test)

nb_map = build_neighborhood_target_stat(train)
if nb_map:
    mean_sp = train["SalePrice"].mean()
    train_fe["NeighborhoodPrice"] = train_fe["Neighborhood"].map(nb_map).fillna(mean_sp)
    test_fe["NeighborhoodPrice"]  = test_fe["Neighborhood"].map(nb_map).fillna(mean_sp)

# ------------------------
# Prepare data
# ------------------------
y = train_fe["SalePrice"].copy()
X = train_fe.drop(columns=["SalePrice", "Id"])
X_test = test_fe.drop(columns=["Id"])

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# ------------------------
# Preprocess + Model
# ------------------------
numeric_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", make_ohe())  # dense OHE to avoid sparse->dense surprises
])

# Let ColumnTransformer use its default sparse_threshold (0.3) — but we output dense anyway
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pre, num_cols),
        ("cat", categorical_pre, cat_cols),
    ]
)

alphas = np.logspace(-3, 4, 20)
ridgecv = RidgeCV(alphas=alphas, cv=5)  # dense-friendly

model = TransformedTargetRegressor(
    regressor=Pipeline(steps=[
        ("prep", preprocessor),
        ("ridge", ridgecv)
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

# ------------------------
# Cross-validation (R^2, MSE, RMSE)
# ------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(model, X, y, scoring="r2", cv=cv)  # keep default n_jobs to limit RAM
cv_mse = -cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=cv)
cv_rmse = np.sqrt(cv_mse)

print("=== 5-fold CV (engineered features) ===")
print(f"R^2:  {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"MSE:  {cv_mse.mean():.2f} ± {cv_mse.std():.2f}")
print(f"RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}")

# ------------------------
# Fit on full train & in-sample metrics
# ------------------------
model.fit(X, y)
best_alpha = model.regressor_.named_steps["ridge"].alpha_
print(f"\nBest Ridge alpha: {best_alpha:.5f}")

y_pred_in = model.predict(X)
r2_in  = r2_score(y, y_pred_in)
mse_in = mean_squared_error(y, y_pred_in)
rmse_in = mean_squared_error(y, y_pred_in, squared=False)

print("\n=== In-sample (fit on full train) ===")
print(f"R^2:  {r2_in:.4f}")
print(f"MSE:  {mse_in:.2f}")
print(f"RMSE: {rmse_in:.2f}")

# ------------------------
# Predict on test & save submission
# ------------------------
test_pred = model.predict(X_test)
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": test_pred})
submission.to_csv("submission.csv", index=False)
print("\nSaved: submission.csv")


In [ ]:
# ==========================================================
# Compare tree models on a fixed validation split (RMSE)
#   - RandomForestRegressor (sklearn)
#   - GradientBoostingRegressor (sklearn)
#   - XGBRegressor (xgboost) with early stopping (constructor)
# ==========================================================
import warnings
warnings.filterwarnings("ignore", message=".*use_inf_as_na.*")
warnings.filterwarnings("ignore", message=".*default of observed=False is deprecated.*")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    has_xgb = True
except Exception:
    has_xgb = False
    print("xgboost not available; skipping XGBRegressor.")

# ------------------------
# Load data
# ------------------------
TRAIN_PATH = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
TEST_PATH  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# ------------------------
# Helpers for robust features
# ------------------------
def safe_get(df, col, default=np.nan):
    return df[col] if col in df.columns else pd.Series(default, index=df.index)

def to_num(s, fill=None):
    out = pd.to_numeric(s, errors="coerce")
    if fill is not None:
        out = out.fillna(fill)
    return out

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Coerced numeric bases
    TotalBsmtSF = to_num(safe_get(out, "TotalBsmtSF"), fill=0)
    FirstFlrSF  = to_num(safe_get(out, "1stFlrSF"), fill=0)
    SecondFlrSF = to_num(safe_get(out, "2ndFlrSF"), fill=0)
    FullBath    = to_num(safe_get(out, "FullBath"), fill=0)
    HalfBath    = to_num(safe_get(out, "HalfBath"), fill=0)
    BsmtFull    = to_num(safe_get(out, "BsmtFullBath"), fill=0)
    BsmtHalf    = to_num(safe_get(out, "BsmtHalfBath"), fill=0)
    YrSold      = to_num(safe_get(out, "YrSold"))
    YearBuilt   = to_num(safe_get(out, "YearBuilt"))
    YearRemod   = to_num(safe_get(out, "YearRemodAdd"))
    OpenPorchSF = to_num(safe_get(out, "OpenPorchSF"), fill=0)
    EnclosedPor = to_num(safe_get(out, "EnclosedPorch"), fill=0)
    ThreeSsn    = to_num(safe_get(out, "3SsnPorch"), fill=0)
    ScreenPorch = to_num(safe_get(out, "ScreenPorch"), fill=0)
    WoodDeckSF  = to_num(safe_get(out, "WoodDeckSF"), fill=0)
    PoolArea    = to_num(safe_get(out, "PoolArea"))
    GarageArea  = to_num(safe_get(out, "GarageArea"))
    GarageYrBlt = to_num(safe_get(out, "GarageYrBlt"))
    GrLivArea   = to_num(safe_get(out, "GrLivArea"))
    LotArea     = to_num(safe_get(out, "LotArea"))
    TotRms      = to_num(safe_get(out, "TotRmsAbvGrd"), fill=0)
    OverallQual = to_num(safe_get(out, "OverallQual"), fill=0)
    OverallCond = to_num(safe_get(out, "OverallCond"), fill=0)

    # Totals & counts
    out["TotalSF"] = TotalBsmtSF + FirstFlrSF + SecondFlrSF
    out["TotalBaths"] = FullBath + 0.5 * HalfBath + BsmtFull + 0.5 * BsmtHalf

    # Age features
    out["HouseAge"] = (YrSold - YearBuilt).fillna(0)
    out["RemodAge"] = (YrSold - YearRemod).fillna(0)
    out["IsRemodeled"] = ((YearRemod - YearBuilt).fillna(-1).gt(0)).astype(int)
    out["IsNew"] = (YrSold == YearBuilt).astype(int)

    # Porches/decks aggregate
    out["TotalPorchSF"] = OpenPorchSF + EnclosedPor + ThreeSsn + ScreenPorch + WoodDeckSF

    # Binary amenities
    out["HasPool"]     = PoolArea.fillna(0).gt(0).astype(int)
    out["Has2ndFloor"] = SecondFlrSF.fillna(0).gt(0).astype(int)
    out["HasBsmt"]     = TotalBsmtSF.fillna(0).gt(0).astype(int)
    out["HasGarage"]   = GarageArea.fillna(0).gt(0).astype(int)

    # Interactions / ratios
    out["QualCond"]  = OverallQual * OverallCond
    out["Qual_LnSF"] = OverallQual * np.log1p(GrLivArea.fillna(0))
    out["BathsPerRoom"] = out["TotalBaths"] / (TotRms + 1.0)

    # LotRatio = GrLivArea / LotArea, avoid /0 or NaN denom
    denom = LotArea.values
    numer = GrLivArea.fillna(0).values
    valid = np.isfinite(denom) & (denom > 0)
    lotratio = np.full_like(numer, np.nan, dtype=float)
    lotratio[valid] = numer[valid] / denom[valid]
    out["LotRatio"] = lotratio

    # GarageAge with fallbacks
    garage_age = (to_num(safe_get(out, "YrSold")) - GarageYrBlt).values.astype(float)
    mask_garage_nan = ~np.isfinite(GarageYrBlt.values)
    mask_has_garage = GarageArea.fillna(0).values > 0
    fallback = (YrSold - YearBuilt).fillna(0).values
    garage_age[np.where(mask_garage_nan & mask_has_garage)] = fallback[np.where(mask_garage_nan & mask_has_garage)]
    garage_age[np.where(mask_garage_nan & ~mask_has_garage)] = 0.0
    garage_age[~np.isfinite(garage_age)] = 0.0
    out["GarageAge"] = garage_age

    # Clean inf
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)
    return out

# Smoothed Neighborhood target stat (built only from TRAIN)
def build_neighborhood_target_stat(train_df: pd.DataFrame, m: float = 50.0):
    if "Neighborhood" not in train_df.columns:
        return {}
    overall = train_df["SalePrice"].mean()
    gb = train_df.groupby("Neighborhood", observed=True)["SalePrice"].agg(["mean", "count"])
    smoothed = (gb["count"] * gb["mean"] + m * overall) / (gb["count"] + m)
    return smoothed.to_dict()

# Apply features
train_fe = add_engineered_features(train)
test_fe  = add_engineered_features(test)

nb_map = build_neighborhood_target_stat(train)
if nb_map:
    overall = train["SalePrice"].mean()
    train_fe["NeighborhoodPrice"] = train_fe["Neighborhood"].map(nb_map).fillna(overall)
    test_fe["NeighborhoodPrice"]  = test_fe["Neighborhood"].map(nb_map).fillna(overall)

# ------------------------
# Prepare X, y and split
# ------------------------
y = train_fe["SalePrice"].copy()
X = train_fe.drop(columns=["SalePrice", "Id"])
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Tree models don't need scaling, just impute + OHE
numeric_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pre, num_cols),
        ("cat", categorical_pre, cat_cols),
    ],
    sparse_threshold=0.0  # dense output
)

# ------------------------
# Define models (XGB with early_stopping in constructor)
# ------------------------
models = {}

models["RandomForest"] = Pipeline(steps=[
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=600,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        oob_score=False
    ))
])

models["GradientBoosting"] = Pipeline(steps=[
    ("prep", preprocessor),
    ("gbr", GradientBoostingRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        random_state=42
    ))
])

if has_xgb:
    models["XGBoost"] = Pipeline(steps=[
        ("prep", preprocessor),
        ("xgb", XGBRegressor(
            n_estimators=2000,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            reg_alpha=0.0,
            objective="reg:squarederror",
            tree_method="hist",   # fast on CPU
            random_state=42,
            n_jobs=-1,
            early_stopping_rounds=100,  # moved here to silence warning
            eval_metric="rmse"
        ))
    ])

# ------------------------
# Train, validate, report
# ------------------------
results = []

for name, pipe in models.items():
    if name == "XGBoost":
        # Manual transform to pass valid set into XGB for early stopping
        Xtr = pipe.named_steps["prep"].fit_transform(X_train, y_train)
        Xva = pipe.named_steps["prep"].transform(X_valid)
        ytr = y_train.values
        yva = y_valid.values

        est = pipe.named_steps["xgb"]
        est.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)

        # Use best_iteration if available
        try:
            y_pred = est.predict(Xva, iteration_range=(0, est.best_iteration + 1))
        except Exception:
            y_pred = est.predict(Xva)
    else:
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_valid)

    rmse = mean_squared_error(y_valid, y_pred, squared=False)
    results.append({"Model": name, "Validation_RMSE": rmse})

# Pretty table
results_df = pd.DataFrame(results).sort_values("Validation_RMSE")
print("\nValidation RMSE by model (lower is better):")
print(results_df.to_string(index=False))

# Keep best model name handy if you want to reuse it
best_name = results_df.iloc[0]["Model"]
print(f"\nBest model on this split: {best_name}")


In [ ]:
# ==========================================================
# End-to-end: build features, compare models, pick best, submit
# Models: RandomForest, GradientBoosting, XGBoost
# ==========================================================
import warnings
warnings.filterwarnings("ignore", message=".*use_inf_as_na.*")
warnings.filterwarnings("ignore", message=".*default of observed=False is deprecated.*")

import numpy as np
import pandas as pd
from copy import deepcopy

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# XGBoost (optional)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print("xgboost not available; skipping XGBRegressor.")

# ------------------------
# Load data
# ------------------------
TRAIN_PATH = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
TEST_PATH  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# ------------------------
# Feature Engineering (NaN-safe)
# ------------------------
def safe_get(df, col, default=np.nan):
    return df[col] if col in df.columns else pd.Series(default, index=df.index)

def to_num(s, fill=None):
    out = pd.to_numeric(s, errors="coerce")
    if fill is not None:
        out = out.fillna(fill)
    return out

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    TotalBsmtSF = to_num(safe_get(out, "TotalBsmtSF"), fill=0)
    FirstFlrSF  = to_num(safe_get(out, "1stFlrSF"), fill=0)
    SecondFlrSF = to_num(safe_get(out, "2ndFlrSF"), fill=0)
    FullBath    = to_num(safe_get(out, "FullBath"), fill=0)
    HalfBath    = to_num(safe_get(out, "HalfBath"), fill=0)
    BsmtFull    = to_num(safe_get(out, "BsmtFullBath"), fill=0)
    BsmtHalf    = to_num(safe_get(out, "BsmtHalfBath"), fill=0)
    YrSold      = to_num(safe_get(out, "YrSold"))
    YearBuilt   = to_num(safe_get(out, "YearBuilt"))
    YearRemod   = to_num(safe_get(out, "YearRemodAdd"))
    OpenPorchSF = to_num(safe_get(out, "OpenPorchSF"), fill=0)
    EnclosedPor = to_num(safe_get(out, "EnclosedPorch"), fill=0)
    ThreeSsn    = to_num(safe_get(out, "3SsnPorch"), fill=0)
    ScreenPorch = to_num(safe_get(out, "ScreenPorch"), fill=0)
    WoodDeckSF  = to_num(safe_get(out, "WoodDeckSF"), fill=0)
    PoolArea    = to_num(safe_get(out, "PoolArea"))
    GarageArea  = to_num(safe_get(out, "GarageArea"))
    GarageYrBlt = to_num(safe_get(out, "GarageYrBlt"))
    GrLivArea   = to_num(safe_get(out, "GrLivArea"))
    LotArea     = to_num(safe_get(out, "LotArea"))
    TotRms      = to_num(safe_get(out, "TotRmsAbvGrd"), fill=0)
    OverallQual = to_num(safe_get(out, "OverallQual"), fill=0)
    OverallCond = to_num(safe_get(out, "OverallCond"), fill=0)

    out["TotalSF"] = TotalBsmtSF + FirstFlrSF + SecondFlrSF
    out["TotalBaths"] = FullBath + 0.5 * HalfBath + BsmtFull + 0.5 * BsmtHalf

    out["HouseAge"] = (YrSold - YearBuilt).fillna(0)
    out["RemodAge"] = (YrSold - YearRemod).fillna(0)
    out["IsRemodeled"] = ((YearRemod - YearBuilt).fillna(-1).gt(0)).astype(int)
    out["IsNew"] = (YrSold == YearBuilt).astype(int)

    out["TotalPorchSF"] = OpenPorchSF + EnclosedPor + ThreeSsn + ScreenPorch + WoodDeckSF

    out["HasPool"]     = PoolArea.fillna(0).gt(0).astype(int)
    out["Has2ndFloor"] = SecondFlrSF.fillna(0).gt(0).astype(int)
    out["HasBsmt"]     = TotalBsmtSF.fillna(0).gt(0).astype(int)
    out["HasGarage"]   = GarageArea.fillna(0).gt(0).astype(int)

    out["QualCond"]  = OverallQual * OverallCond
    out["Qual_LnSF"] = OverallQual * np.log1p(GrLivArea.fillna(0))
    out["BathsPerRoom"] = out["TotalBaths"] / (TotRms + 1.0)

    denom = LotArea.values
    numer = GrLivArea.fillna(0).values
    valid = np.isfinite(denom) & (denom > 0)
    lotratio = np.full_like(numer, np.nan, dtype=float)
    lotratio[valid] = numer[valid] / denom[valid]
    out["LotRatio"] = lotratio

    garage_age = (to_num(safe_get(out, "YrSold")) - GarageYrBlt).values.astype(float)
    mask_garage_nan = ~np.isfinite(GarageYrBlt.values)
    mask_has_garage = GarageArea.fillna(0).values > 0
    fallback = (YrSold - YearBuilt).fillna(0).values
    garage_age[np.where(mask_garage_nan & mask_has_garage)] = fallback[np.where(mask_garage_nan & mask_has_garage)]
    garage_age[np.where(mask_garage_nan & ~mask_has_garage)] = 0.0
    garage_age[~np.isfinite(garage_age)] = 0.0
    out["GarageAge"] = garage_age

    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)
    return out

def build_neighborhood_target_stat(train_df: pd.DataFrame, m: float = 50.0):
    if "Neighborhood" not in train_df.columns:
        return {}
    overall = train_df["SalePrice"].mean()
    gb = train_df.groupby("Neighborhood", observed=True)["SalePrice"].agg(["mean", "count"])
    smoothed = (gb["count"] * gb["mean"] + m * overall) / (gb["count"] + m)
    return smoothed.to_dict()

# Apply features
train_fe = add_engineered_features(train)
test_fe  = add_engineered_features(test)

nb_map = build_neighborhood_target_stat(train)
if nb_map:
    overall = train["SalePrice"].mean()
    train_fe["NeighborhoodPrice"] = train_fe["Neighborhood"].map(nb_map).fillna(overall)
    test_fe["NeighborhoodPrice"]  = test_fe["Neighborhood"].map(nb_map).fillna(overall)

# X / y / split
y = train_fe["SalePrice"].copy()
X = train_fe.drop(columns=["SalePrice", "Id"])
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Preprocessor (trees: impute + OHE)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_pre = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_pre = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pre, num_cols),
        ("cat", categorical_pre, cat_cols),
    ],
    sparse_threshold=0.0
)

# Define models
models = {}
models["RandomForest"] = Pipeline(steps=[
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=600, max_depth=None, max_features="sqrt",
        min_samples_split=2, min_samples_leaf=1,
        n_jobs=-1, random_state=42
    ))
])

models["GradientBoosting"] = Pipeline(steps=[
    ("prep", preprocessor),
    ("gbr", GradientBoostingRegressor(
        n_estimators=1200, learning_rate=0.03, max_depth=3,
        subsample=0.8, random_state=42
    ))
])

if HAS_XGB:
    models["XGBoost"] = Pipeline(steps=[
        ("prep", preprocessor),
        ("xgb", XGBRegressor(
            n_estimators=2000, learning_rate=0.03, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, reg_alpha=0.0,
            objective="reg:squarederror",
            tree_method="hist", random_state=42, n_jobs=-1,
            # early stopping for validation phase only
            early_stopping_rounds=100, eval_metric="rmse"
        ))
    ])

# Train/validate
results = []
for name, pipe in models.items():
    if name == "XGBoost":
        Xtr = pipe.named_steps["prep"].fit_transform(X_train, y_train)
        Xva = pipe.named_steps["prep"].transform(X_valid)
        ytr = y_train.values
        yva = y_valid.values

        est = pipe.named_steps["xgb"]
        est.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)

        try:
            y_pred = est.predict(Xva, iteration_range=(0, est.best_iteration + 1))
        except Exception:
            y_pred = est.predict(Xva)
    else:
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_valid)

    rmse = mean_squared_error(y_valid, y_pred, squared=False)
    results.append({"Model": name, "Validation_RMSE": rmse})

results_df = pd.DataFrame(results).sort_values("Validation_RMSE")
print("\nValidation RMSE by model (lower is better):")
print(results_df.to_string(index=False))

best_name = results_df.iloc[0]["Model"]
print(f"\nBest model on this split: {best_name}")

# ------------------------
# Refit best on ALL data & predict test
# ------------------------
X_test = test_fe.drop(columns=["Id"])
test_ids = test_fe["Id"].copy()

if best_name == "XGBoost":
    prep = models["XGBoost"].named_steps["prep"]
    est  = models["XGBoost"].named_steps["xgb"]

    X_full_tr = prep.fit_transform(X, y)
    X_test_tr = prep.transform(X_test)

    est_full = deepcopy(est)
    est_full.set_params(early_stopping_rounds=None)  # disable for full fit
    est_full.fit(X_full_tr, y.values, verbose=False)

    y_test_pred = est_full.predict(X_test_tr)
else:
    best_pipe = models[best_name]
    best_pipe.fit(X, y)
    y_test_pred = best_pipe.predict(X_test)

# Save submission
submission = pd.DataFrame({"Id": test_ids, "SalePrice": y_test_pred})
submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv")
print(submission.head())
